<a href="https://colab.research.google.com/github/DoniaGabal/Advanced_Rag_system/blob/main/Snake_Game_Environment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
#  Snake Game Environment
#  Grid: configurable size (default 10x10, supports 12x12)
#  Actions: 0=Straight, 1=Turn Left, 2=Turn Right
# ============================================================

# Directions as (row_delta, col_delta)
UP    = (-1,  0)
DOWN  = ( 1,  0)
LEFT  = ( 0, -1)
RIGHT = ( 0,  1)

# Turn left / right mappings (relative to current direction)
TURN_LEFT = {
    UP:    LEFT,
    LEFT:  DOWN,
    DOWN:  RIGHT,
    RIGHT: UP,
}
TURN_RIGHT = {
    UP:    RIGHT,
    RIGHT: DOWN,
    DOWN:  LEFT,
    LEFT:  UP,
}

class SnakeEnv:
    """
    Snake environment for DQN training.

    Parameters:
        grid_size : int  — size of the square grid (10 or 12, default 10)
        obstacles : bool — enable fixed obstacles (default True)

    Usage:
        env = SnakeEnv(grid_size=10, obstacles=True)
        state = env.reset()
        state, reward, done, info = env.step(action)  # action in {0,1,2}
    """

    BASE_LOOP_LIMIT = 100   # scales with grid size

    def __init__(self, grid_size=10, obstacles=True):
        self.GRID_SIZE     = grid_size
        self.use_obstacles = obstacles

        # Obstacles scale with grid size (placed at ~30% and ~70% of grid)
        a = grid_size // 3
        b = grid_size - a - 1
        self.obstacles = [(a, a), (a, a+1), (b, b), (b, b+1)] if obstacles else []

        self.reset()

    # ----------------------------------------------------------
    # reset — call at the start of every episode
    # ----------------------------------------------------------
    def reset(self):
        """Reset the environment and return the initial state (11 elements)."""
        mid = self.GRID_SIZE // 2
        self.snake = [
            (mid, mid),
            (mid, mid - 1),
            (mid, mid - 2),
        ]
        self.direction        = RIGHT
        self.score            = 0
        self.steps            = 0
        self.steps_since_food = 0
        self.speed_level      = 1   # BONUS: starts at 1, increases every 3 food

        self._spawn_food()
        return self._get_state()

    # ----------------------------------------------------------
    # step — main function called every timestep
    # ----------------------------------------------------------
    def step(self, action):
        """
        Execute one action and return (state, reward, done, info).

        action : 0=straight | 1=turn left | 2=turn right
        info   : dict with score, speed_level, snake_length, steps
        """
        self.steps += 1
        self.steps_since_food += 1

        # 1. Update direction
        if action == 1:
            self.direction = TURN_LEFT[self.direction]
        elif action == 2:
            self.direction = TURN_RIGHT[self.direction]

        # 2. Compute new head position
        head_r, head_c = self.snake[0]
        dr, dc = self.direction
        new_head = (head_r + dr, head_c + dc)

        # 3. Collision check BEFORE moving
        if self._is_collision(new_head):
            return self._get_state(), -10, True, self._info()

        # 4. Move snake forward
        self.snake.insert(0, new_head)

        # 5. Food check
        if new_head == self.food:
            self.score            += 1
            self.steps_since_food  = 0
            reward                 = +10
            self._spawn_food()

            # BONUS: speed increases every 3 food items eaten
            self.speed_level = 1 + (self.score // 3)
            # Snake grows automatically (no pop)
        else:
            self.snake.pop()   # remove tail → same length
            reward = -0.1

        # 6. Loop detection — limit scales with grid size
        loop_limit = self.BASE_LOOP_LIMIT * (self.GRID_SIZE // 10)
        if self.steps_since_food > loop_limit:
            return self._get_state(), -10, True, self._info()

        return self._get_state(), reward, False, self._info()

    def _info(self):
        """Extra info returned with every step (useful for logging)."""
        return {
            "score":        self.score,
            "speed_level":  self.speed_level,    # BONUS
            "snake_length": len(self.snake),
            "steps":        self.steps,
        }

    # ----------------------------------------------------------
    # _get_state — compute the 11-element state vector
    # ----------------------------------------------------------
    def _get_state(self):
        """
        Returns a list of 11 binary values:

        [danger_straight, danger_left, danger_right,
         food_left, food_right, food_up, food_down,
         moving_left, moving_right, moving_up, moving_down]

        All values are 0 or 1.
        NOTE: danger is relative to the snake's CURRENT direction.
        """
        head = self.snake[0]
        head_r, head_c = head

        # Positions one step ahead in each relative direction
        straight_pos = self._pos_in_direction(head, self.direction)
        left_pos      = self._pos_in_direction(head, TURN_LEFT[self.direction])
        right_pos     = self._pos_in_direction(head, TURN_RIGHT[self.direction])

        food_r, food_c = self.food

        state = [
            # --- Danger (1 = danger, 0 = safe) ---
            int(self._is_collision(straight_pos)),   # danger straight
            int(self._is_collision(left_pos)),        # danger left
            int(self._is_collision(right_pos)),       # danger right

            # --- Food direction relative to head ---
            int(food_c < head_c),   # food is to the LEFT
            int(food_c > head_c),   # food is to the RIGHT
            int(food_r < head_r),   # food is UP   (row decreases going up)
            int(food_r > head_r),   # food is DOWN

            # --- Current movement direction (one-hot) ---
            int(self.direction == LEFT),
            int(self.direction == RIGHT),
            int(self.direction == UP),
            int(self.direction == DOWN),
        ]

        return state  # length = 11

    # ----------------------------------------------------------
    # _is_collision — check if a position causes game over
    # ----------------------------------------------------------
    def _is_collision(self, pos):
        """Return True if pos is a wall, body segment, or obstacle."""
        r, c = pos

        # Hit wall
        if r < 0 or r >= self.GRID_SIZE or c < 0 or c >= self.GRID_SIZE:
            return True

        # Hit own body (skip head — index 0 — because it moves away)
        if pos in self.snake[1:]:
            return True

        # Hit obstacle (bonus)
        if pos in self.obstacles:
            return True

        return False

    # ----------------------------------------------------------
    # _spawn_food — place food randomly (not on snake or obstacles)
    # ----------------------------------------------------------
    def _spawn_food(self):
        """Randomly place food somewhere not occupied."""
        all_cells = [
            (r, c)
            for r in range(self.GRID_SIZE)
            for c in range(self.GRID_SIZE)
        ]
        occupied = set(self.snake) | set(self.obstacles)
        free_cells = [cell for cell in all_cells if cell not in occupied]

        if free_cells:
            self.food = random.choice(free_cells)
        else:
            # Edge case: board is full → episode ends (extremely rare)
            self.food = None

    # ----------------------------------------------------------
    # _pos_in_direction — helper: one step in a direction
    # ----------------------------------------------------------
    def _pos_in_direction(self, pos, direction):
        """Return the position one step ahead in the given direction."""
        r, c = pos
        dr, dc = direction
        return (r + dr, c + dc)

    # ----------------------------------------------------------
    # render — print grid to terminal (for debugging)
    # ----------------------------------------------------------
    def render(self):
        """Print a text representation of the current state (for debugging)."""
        grid = [['.' for _ in range(self.GRID_SIZE)]
                for _ in range(self.GRID_SIZE)]

        for (r, c) in self.obstacles:
            grid[r][c] = '#'
        for (r, c) in self.snake[1:]:
            grid[r][c] = 'o'
        hr, hc = self.snake[0]
        grid[hr][hc] = 'H'
        if self.food:
            fr, fc = self.food
            grid[fr][fc] = '*'

        dir_symbol = {RIGHT: '→', LEFT: '←', UP: '↑', DOWN: '↓'}
        print(f"\nGrid:{self.GRID_SIZE}x{self.GRID_SIZE} | "
              f"Score:{self.score} | Steps:{self.steps} | "
              f"Speed Level:{self.speed_level} | "       # BONUS
              f"Length:{len(self.snake)} | "
              f"Dir:{dir_symbol.get(self.direction,'?')}")
        print("+" + "-" * self.GRID_SIZE + "+")
        for row in grid:
            print("|" + "".join(row) + "|")
        print("+" + "-" * self.GRID_SIZE + "+")


# ============================================================
#  Quick test — run:  python environment.py
# ============================================================
if __name__ == "__main__":
    print("=" * 52)
    print("  TEST: Snake Environment — 10x10 with obstacles")
    print("=" * 52)

    env   = SnakeEnv(grid_size=10, obstacles=True)
    state = env.reset()
    print("Initial state (11 elements):", state)
    print("State length:", len(state))
    env.render()

    for i in range(10):
        action = random.randint(0, 2)
        state, reward, done, info = env.step(action)
        print(f"Step {i+1:2d} | Action:{action} | Reward:{reward:+.1f} | "
              f"Done:{done} | Speed Level:{info['speed_level']}")
        if done:
            print("  → Episode ended, resetting...")
            env.reset()

    env.render()

  TEST: Snake Environment — 10x10 with obstacles
Initial state (11 elements): [0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0]
State length: 11

Grid:10x10 | Score:0 | Steps:0 | Speed Level:1 | Length:3 | Dir:→
+----------+
|..........|
|..........|
|..........|
|...##.....|
|..........|
|...ooH....|
|...*..##..|
|..........|
|..........|
|..........|
+----------+
Step  1 | Action:0 | Reward:-0.1 | Done:False | Speed Level:1
Step  2 | Action:0 | Reward:-0.1 | Done:False | Speed Level:1
Step  3 | Action:2 | Reward:-10.0 | Done:True | Speed Level:1
  → Episode ended, resetting...
Step  4 | Action:2 | Reward:-0.1 | Done:False | Speed Level:1
Step  5 | Action:0 | Reward:-0.1 | Done:False | Speed Level:1
Step  6 | Action:0 | Reward:-0.1 | Done:False | Speed Level:1
Step  7 | Action:2 | Reward:-0.1 | Done:False | Speed Level:1
Step  8 | Action:1 | Reward:-0.1 | Done:False | Speed Level:1
Step  9 | Action:2 | Reward:-0.1 | Done:False | Speed Level:1
Step 10 | Action:0 | Reward:-0.1 | Done:False | Speed Leve